# Session 32: How Models Learn: Gradient Descent

Data Science with Python, Level 2, Stage 5: ML Fundamentals

## Learning Objectives

By the end of this session you will be able to:

1. Measure a model's error with a loss function.
2. Explain what a gradient tells us.
3. Write gradient descent from scratch.
4. Choose a learning rate.
5. Explain why feature scaling speeds up learning.
6. Connect your code to scikit-learn's models.

## Where We Are

In Session 31, `model.fit` found the best line in an instant. But what does **best** mean, and how does a computer find it? The answer is the engine inside almost every machine learning model, from linear regression to the neural networks behind modern AI: **gradient descent**.

The idea fits in one sentence: measure how wrong the model is, find which direction makes it less wrong, take a small step that way, and repeat. Today you will write it yourself in about ten lines of NumPy.

## Setup

Run this cell first. We model the math score `y` from weekly study hours `x`: one feature keeps every step visible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, SGDRegressor

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
exams = pd.read_csv(DATA_URL + "exam_scores.csv")
x = exams["study_hours_week"].to_numpy()
y = exams["math"].to_numpy(dtype=float)

## 1. Measuring Error with a Loss Function

A line predicts `y_hat = w * x + b`, where `w` is the slope (also called the **weight**) and `b` the intercept (the **bias**). To compare lines we need one number for the total error. Linear regression uses the **mean squared error** (MSE): the average of the squared residuals. Squaring makes every error positive and punishes big errors more.

In [ ]:
def mse(w, b):
    y_hat = w * x + b
    return np.mean((y - y_hat) ** 2)

for w, b in [(0, 57), (1, 50), (1.5, 46), (3, 30)]:
    print(f"w = {w}, b = {b}: loss = {mse(w, b):.1f}")

In [ ]:
ws = np.linspace(-1, 4, 100)
losses = [mse(w, 46.2) for w in ws]

plt.plot(ws, losses)
plt.title("Loss for different slopes (b fixed at 46.2)")
plt.xlabel("Slope w")
plt.ylabel("Mean squared error")
plt.show()

## 2. The Gradient

The **gradient** is the slope of the loss with respect to each parameter. If increasing `w` makes the loss go up, the gradient for `w` is positive, so we should **decrease** `w`. For MSE the gradients have simple formulas:

- for `w`: `-2 * mean(x * (y - y_hat))`
- for `b`: `-2 * mean(y - y_hat)`

You do not need to derive them. What matters is the rule: **step against the gradient**.

In [ ]:
def gradients(w, b):
    error = y - (w * x + b)
    dw = -2 * np.mean(x * error)
    db = -2 * np.mean(error)
    return dw, db

for w in [0.5, 3.0]:
    dw, db = gradients(w, 46.2)
    print(f"w = {w}: gradient for w {dw:.1f}, for b {db:.1f}")

## 3. Gradient Descent

In [ ]:
w, b = 0.0, 0.0
learning_rate = 0.002
history = []

for step in range(20_000):
    dw, db = gradients(w, b)
    w -= learning_rate * dw
    b -= learning_rate * db
    if step % 2000 == 0:
        history.append(mse(w, b))

print(f"w = {w:.3f}, b = {b:.2f}, loss = {mse(w, b):.1f}")

In [ ]:
sk = LinearRegression().fit(x.reshape(-1, 1), y)
print(f"scikit-learn: w = {sk.coef_[0]:.3f}, b = {sk.intercept_:.2f}")

## 4. The Learning Rate

In [ ]:
def train(lr, steps=300, w=0.0, b=46.2):
    losses = []
    for _ in range(steps):
        dw, db = gradients(w, b)
        w, b = w - lr * dw, b - lr * db
        losses.append(mse(w, b))
    return losses

for lr in [0.0005, 0.005, 0.017]:
    plt.plot(train(lr), label=f"learning rate {lr}")
plt.yscale("log")
plt.ylim(100, 5000)
plt.title("Loss during training")
plt.xlabel("Step")
plt.legend()
plt.show()

## 5. Feature Scaling

When features have very different sizes, for example area in square metres and distance in kilometres, the loss valley becomes long and narrow, and gradient descent zigzags slowly. **Standardising** each feature, subtracting its mean and dividing by its standard deviation as in Session 26, makes the valley round and training much faster.

In [ ]:
x_std = (x - x.mean()) / x.std()

w, b = 0.0, 0.0
for step in range(200):
    error = y - (w * x_std + b)
    w += 0.1 * 2 * np.mean(x_std * error)
    b += 0.1 * 2 * np.mean(error)

print(f"After 200 steps: w = {w:.3f}, b = {b:.2f}")
print("Slope in original units:", round(w / x.std(), 3))

## 6. Gradient Descent in scikit-learn

In [ ]:
sgd = SGDRegressor(learning_rate="constant", eta0=0.01, max_iter=1000, random_state=0)
sgd.fit(x_std.reshape(-1, 1), y)

print(f"SGD: w = {sgd.coef_[0]:.2f}, b = {sgd.intercept_[0]:.2f}")
print(f"Ours: w = {w:.2f}, b = {b:.2f}")

| Model | What is learned | Session |
| --- | --- | --- |
| Linear regression (SGD) | One weight per feature | Today |
| Logistic regression | Weights for a probability | 33 |
| Gradient boosting | Trees added along the gradient | 38 |
| Neural networks | Millions of weights | Beyond this course |

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Loss explodes | A learning rate that is too large | Divide it by 10 and plot the loss again. |
| Never finishes | A learning rate that is far too small | Multiply it by 10, or scale the features. |
| Slow and zigzag | Features on very different scales | Standardise the features before training. |
| Wrong direction | Adding the gradient instead of subtracting it | Update with `w = w - lr * gradient`. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Loss of the Mean

What is the MSE of always predicting the mean math score? Compare it with the loss of the best line. How much did the line reduce the error?

In [ ]:
# Write your code below

### In-Class Exercise 2: Plot the Descent

Run gradient descent on the standardised feature for 50 steps with learning rate 0.1, store the loss at every step, and plot it.

In [ ]:
# Write your code below

### In-Class Exercise 3: Find the Breaking Point

With the standardised feature, try learning rates 0.5, 0.9, 1.0 and 1.1 for 50 steps. Which ones converge and which explode?

In [ ]:
# Write your code below

### In-Class Exercise 4: Two Features

Extend the loop to two standardised features, study hours and sleep hours, with one weight each. Compare your weights with `LinearRegression` on the same standardised features.

In [ ]:
# Write your code below

## Summary

- A loss function measures how wrong a model is; MSE is common.
- The gradient points uphill; we step the other way.
- Gradient descent: `w = w - learning_rate * gradient`, repeated.
- The learning rate controls the step size: test a few.
- Standardised features make training much faster.
- SGD uses small batches; the same idea trains most models.

## What's Next

So far the target was a number. Next session the target is a category: will this customer leave, yes or no? You will meet logistic regression, which predicts probabilities.